In [1]:
import pandas as pd

df = pd.read_csv('../dataset/email_spam_indo.csv')
df.head()

,Kategori,Pesan
0,spam,Secara alami tak tertahankan identitas perusah...
1,spam,Fanny Gunslinger Perdagangan Saham adalah Merr...
2,spam,Rumah -rumah baru yang luar biasa menjadi muda...
3,spam,4 Permintaan Khusus Pencetakan Warna Informasi...
4,spam,"Jangan punya uang, dapatkan CD perangkat lunak..."


In [2]:
df = df.rename(columns={
    'Pesan': 'text',
    'Kategori': 'label'
})

In [3]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'(dikeret oleh|ditahan oleh|ect pada|subjek:).*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\w+\s+(com|net|org|co|id)\b', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'^(re|fw|fwd):[^.?!\n]*', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head()

,text,clean_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...


In [4]:
NORMALIZATION_DICT = {
    # 🔤 Singkatan umum
    "gk": "tidak",
    "d": "di",
    "ga": "tidak",
    "gak": "tidak",
    "nggak": "tidak",
    "tdk": "tidak",
    "dr": "dari",
    "dgn": "dengan",
    "utk": "untuk",
    "yg": "yang",
    "jg": "juga",
    "krn": "karena",
    "blm": "belum",
    "sdh": "sudah",
    "aja": "saja",
    "trs": "terus",
    "bgt": "banget",
    "dlm": "dalam",

    # 💰 Kata khas spam / promo
    "rp": "rupiah",
    "jt": "juta",
    "rb": "ribu",
    "cashback": "cashback",
    "promo": "promosi",
    "diskon": "diskon",
    "gratis": "gratis",
    "hadiah": "hadiah",
    "menang": "menang",
    "undian": "undian",

    # 📱 Variasi kata penting
    "tlp": "telepon",
    "telp": "telepon",
    "hp": "handphone",
    "no": "nomor",
    "rek": "rekening",
    "an": "atas nama",

    # ⚡ Variasi penulisan spam
    "selamat!!!": "selamat",
    "selamat!!": "selamat",
    "selamat!": "selamat",
    "trnsfer": "transfer",
    "trf": "transfer",

    # 🧩 Variasi angka ke huruf (leetspeak)
    "4nda": "anda",
    "4pa": "apa",
    "1ni": "ini",
    "k4mu": "kamu",

    # 🪤 Kata clickbait spam
    "klik": "klik",
    "segera": "segera",
    "buruan": "cepat",
    "cepat": "cepat",
    "terbatas": "terbatas",
    "khusus": "khusus",

    # 🎯 Kata inti spam
    "free": "gratis",
    "win": "menang",
    "winner": "pemenang",
    "prize": "hadiah",
    "bonus": "bonus",
    "gift": "hadiah",
    "reward": "hadiah",

    # 💰 Finansial / uang
    "cash": "uang",
    "money": "uang",
    "credit": "kredit",
    "loan": "pinjaman",
    "bank": "bank",
    "transfer": "transfer",
    "account": "akun",
    "sspecial": "spesial",
    "balance": "saldo",

    # 📢 Promosi / marketing
    "promo": "promosi",
    "offer": "penawaran",
    "deal": "penawaran",
    "discount": "diskon",
    "sale": "diskon",
    "limited": "terbatas",
    "exclusive": "eksklusif",

    # ⚡ Call to action (penting banget buat spam)
    "click": "klik",
    "claim": "klaim",
    "buy": "beli",
    "order": "pesan",
    "register": "daftar",
    "subscribe": "langganan",
    "join": "gabung",
    "verify": "verifikasi",
    "confirm": "konfirmasi",

    # ⏰ urgency
    "urgent": "segera",
    "now": "sekarang",
    "today": "hari ini",
    "instant": "instan",

    # 🖥️ teknologi / produk
    "software": "perangkat lunak",
    "system": "sistem",
    "update": "pembaruan",
    "download": "unduh",
    "install": "instal",

    # 📧 email umum
    "hello": "halo",
    "dear": "halo",
    "sir": "bapak",
    "madam": "ibu",

    # 🧩 lain-lain yang sering muncul
    "service": "layanan",
    "customer": "pelanggan",
    "support": "dukungan",
    "information": "informasi",
    "message": "pesan"
}
def normalize_text(text):
    words = text.split()
    normalized_words = [NORMALIZATION_DICT.get(word, word) for word in words]
    return ' '.join(normalized_words)

df['normalized_text'] = df['clean_text'].apply(normalize_text)
df[['text', 'clean_text','normalized_text']].head()


,text,clean_text,normalized_text
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham adalah merr...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru yang luar biasa menjadi mudah...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [5]:
!pip install Sastrawi
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# ambil stopword default
factory = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

# fungsi hapus stopword
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopwords])

# apply ke data
df['stopwords'] = df['normalized_text'].apply(remove_stopwords)

# lihat hasil
df[['text', 'normalized_text', 'stopwords']].head()

,text,normalized_text,stopwords
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...


In [7]:
# tokenization
df['tokens'] = df['stopwords'].apply(lambda x: x.split())

# lihat hasil
df[['text', 'clean_text', 'stopwords', 'tokens']].head()

,text,clean_text,stopwords,tokens
0,Secara alami tak tertahankan identitas perusah...,secara alami tak tertahankan identitas perusah...,alami tak tertahankan identitas perusahaan san...,"[alami, tak, tertahankan, identitas, perusahaa..."
1,Fanny Gunslinger Perdagangan Saham adalah Merr...,fanny gunslinger perdagangan saham adalah merr...,fanny gunslinger perdagangan saham merrill muz...,"[fanny, gunslinger, perdagangan, saham, merril..."
2,Rumah -rumah baru yang luar biasa menjadi muda...,rumah rumah baru yang luar biasa menjadi mudah...,rumah rumah baru luar biasa menjadi mudah menu...,"[rumah, rumah, baru, luar, biasa, menjadi, mud..."
3,4 Permintaan Khusus Pencetakan Warna Informasi...,permintaan khusus pencetakan warna informasi t...,permintaan khusus pencetakan warna informasi t...,"[permintaan, khusus, pencetakan, warna, inform..."
4,"Jangan punya uang, dapatkan CD perangkat lunak...",jangan punya uang dapatkan cd perangkat lunak ...,jangan punya uang dapatkan cd perangkat lunak ...,"[jangan, punya, uang, dapatkan, cd, perangkat,..."


In [8]:
!pip install pycaret scikit-learn gensim


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# Model Word2Vec butuh data berformat list kata per kalimat (ada di df['tokens'])
# Kita buat 100 fitur/dimensi vektor untuk setiap kalimat
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=100, window=5, min_count=2, workers=4)

# Fungsi pembuat Sentence/Document Vector (Rata-rata fitur vektor dari kumpulan kata)
def document_vector(doc):
    valid_words = [word for word in doc if word in w2v_model.wv.key_to_index]
    if len(valid_words) == 0:
        return np.zeros(100)
    return np.mean(w2v_model.wv[valid_words], axis=0)

X_w2v = np.array(df['tokens'].apply(document_vector).tolist())

# Susun ke bentuk DataFrame untuk PyCaret
feature_names = [f"w2v_{i}" for i in range(100)]
df_pycaret = pd.DataFrame(X_w2v, columns=feature_names)
df_pycaret['label'] = df['label'].values

df_pycaret.head()

,w2v_0,w2v_1,w2v_2,w2v_3,w2v_4,w2v_5,w2v_6,w2v_7,w2v_8,w2v_9,...,w2v_91,w2v_92,w2v_93,w2v_94,w2v_95,w2v_96,w2v_97,w2v_98,w2v_99,label
0,0.062616,0.852241,0.542056,-0.068858,-0.250792,-0.643764,0.115303,0.699246,-0.132693,-0.684864,...,0.081326,0.264982,0.459792,1.100665,0.099204,0.845415,-0.572950,0.329304,-0.273980,spam
1,-0.005855,0.375089,0.315938,-0.031259,-0.132917,-0.461248,0.133113,0.535756,-0.180179,-0.402832,...,-0.003439,0.192877,0.205998,0.602202,0.095859,0.323144,-0.431961,0.304939,-0.158806,spam
2,0.097316,0.692441,0.492338,-0.103231,-0.222280,-0.701223,0.126632,0.845231,-0.209037,-0.645053,...,0.040314,0.258201,0.528277,1.061614,0.129379,0.682857,-0.639576,0.348959,-0.306694,spam
3,0.108594,0.297771,0.164841,-0.310617,0.409755,-1.095954,-0.299441,1.622640,-0.065684,0.048288,...,0.259571,0.718269,0.583605,1.116216,0.433280,-0.049076,-0.331824,-0.057768,-0.137450,spam
4,0.142632,0.592080,0.279907,-0.004921,-0.226180,-0.302473,0.342426,0.961478,-0.628749,-0.205387,...,0.291008,0.337070,0.539857,1.218750,0.229793,0.376225,-0.209806,0.442308,-0.678148,spam


In [10]:
from pycaret.classification import ClassificationExperiment

# Inisialisasi Environment PyCaret berbasis Object Khusus untuk W2V
exp_w2v = ClassificationExperiment()
clf_setup = exp_w2v.setup(data=df_pycaret, target='label', session_id=42, use_gpu=False)

,Description,Value
0,Session id,42
1,Target,label
2,Target type,Binary
3,Target mapping,"ham: 0, spam: 1"
4,Original data shape,"(2636, 101)"
5,Transformed data shape,"(2636, 101)"
6,Transformed train set shape,"(1845, 101)"
7,Transformed test set shape,"(791, 101)"
8,Numeric features,100
9,Preprocess,True


In [11]:
# Membandingkan model SVM, Naive Bayes (nb), dan Logistic Regression (lr)
best_model = exp_w2v.compare_models(include=['svm', 'nb', 'lr'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
svm,SVM - Linear Kernel,0.9176,0.9734,0.9176,0.9241,0.9171,0.8349,0.8414,0.7380
lr,Logistic Regression,0.9003,0.9602,0.9003,0.9018,0.9001,0.7999,0.8016,0.0290
nb,Naive Bayes,0.7458,0.8799,0.7458,0.7816,0.7351,0.4840,0.5216,0.3110
